# Recommender system using tfidf

https://www.nlplanet.org/course-practical-nlp/01-intro-to-nlp/10a-recsys-tfidf

In [1]:
from huggingface_hub import hf_hub_download

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse

Download dataset

In [2]:
# download dataset of Medium articles from 
# https://huggingface.co/datasets/fabiochiu/medium-articles
df_articles = pd.read_csv(
  hf_hub_download("fabiochiu/medium-articles", repo_type="dataset", filename="medium_articles.csv")
)

# There are 192,368 articles in total, but let's keep only 10,000 of them to
# make computations faster
df_articles = df_articles[:10000].reset_index(drop=True)

df_articles.head()

,title,text,url,authors,timestamp,tags
0,Mental Note Vol. 24,Photo by Josh Riemer on Unsplash\n\nMerry Chri...,https://medium.com/invisible-illness/mental-no...,['Ryan Fan'],2020-12-26 03:38:10.479000+00:00,"['Mental Health', 'Health', 'Psychology', 'Sci..."
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the cu...,https://medium.com/age-of-awareness/how-the-pa...,['Simon Spichak'],2020-09-23 22:10:17.126000+00:00,"['Mental Health', 'Coronavirus', 'Science', 'P..."
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can chang...,https://medium.com/neodotlife/mind-your-nose-f...,[],2020-10-10 20:17:37.132000+00:00,"['Biotechnology', 'Neuroscience', 'Brain', 'We..."
3,The 4 Purposes of Dreams,Passionate about the synergy between science a...,https://medium.com/science-for-real/the-4-purp...,['Eshan Samaranayake'],2020-12-21 16:05:19.524000+00:00,"['Health', 'Neuroscience', 'Mental Health', 'P..."
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage...",https://medium.com/live-your-life-on-purpose/s...,['Rishav Sinha'],2020-02-26 00:01:01.576000+00:00,"['Brain', 'Health', 'Development', 'Psychology..."


Using the tfidf vectoriser

In [3]:
# apply the TfidfVectorizer to the corpus
corpus = df_articles["text"]
vectorizer = TfidfVectorizer()
corpus_vectorized = vectorizer.fit_transform(corpus)
print(corpus_vectorized.shape)
corpus_vectorized[:2]

(10000, 110038)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 919 stored elements and shape (2, 110038)>

Representing users with vectors

In [4]:
def get_sparse_user_vector_for_tag(df_articles, tag, corpus_vectorized, n_articles=3):
  # get the indices of "n_articles" random articles with the "tag" tag
  df_articles_data_science = df_articles[df_articles["tags"].apply(lambda l: tag in eval(l))]
  read_articles_indices = df_articles_data_science.sample(n=n_articles).index.values.tolist()
  read_articles_indices = np.array(read_articles_indices)
  df_read_articles = df_articles.loc[read_articles_indices]

  # compute user vector as the average of the vectors of the read articles
  read_articles_rows = []
  for idx in read_articles_indices:
    article_row = corpus_vectorized.getrow(idx).toarray()[0]
    read_articles_rows.append(article_row)
  read_articles_rows = np.array(read_articles_rows)
  user_vector_dense = np.average(read_articles_rows, axis=0).reshape((1, -1))
  user_vector = sparse.csr_matrix(user_vector_dense)

  return user_vector, df_read_articles

# suppose the user has read three articles about data science
user_vector, df_read_articles = get_sparse_user_vector_for_tag(df_articles, "Data Science",
                                             corpus_vectorized, n_articles=3)
print(user_vector.shape)

(1, 110038)


In [8]:
# checking that the read articles is about data science
df_read_articles["title"].values

array(['Integration in elementary terms',
       'PyTorch Lightning 0.7.1 Release and Venture Funding',
       'The NLP Cypher | 12.20.20'], dtype=object)

Compute similarities between user and articles

In [9]:
# compute scores as the dot product between the query vector
# and the documents vectors
scores = user_vector.dot(corpus_vectorized.transpose())
scores_array = scores.toarray()[0]
print(scores_array.shape)

(10000,)


Show results

In [11]:
# retrieve the top_n articles with the highest scores and show them
def show_best_results(df_articles, scores_array, top_n=10):
  sorted_indices = scores_array.argsort()[::-1]
  for position, idx in enumerate(sorted_indices[:top_n]):
    row = df_articles.iloc[idx]
    title = row["title"]
    score = scores_array[idx]
    print(f"{position + 1} [score = {score}]: {title}")

show_best_results(df_articles, scores_array)

1 [score = 0.41687142948551675]: The NLP Cypher | 12.20.20
2 [score = 0.41334622827590833]: PyTorch Lightning 0.7.1 Release and Venture Funding
3 [score = 0.4107304321946604]: Integration in elementary terms
4 [score = 0.28544727074437093]: How Entrepreneurs Can Thrive in a New Era of Uncertainty
5 [score = 0.28223868797850277]: Tools Of The Best: Software Development Edition
6 [score = 0.26983403382821786]: How to migrate your company to a new product overnight
7 [score = 0.26551951881253305]: Some Thoughts About the Cognitive Revolution We Live In
8 [score = 0.26499471961511817]: Revisiting six memos
9 [score = 0.2644772168050958]: Big Data’s Role in Creating Customer-Centric Business Intelligence
10 [score = 0.26294568295056536]: Nicholas Bloom on Management, Productivity, and Scientific Progress (Ep. 102)


Try a different one

In [12]:
# suppose the user has read three articles about Computer Vision
user_vector, _ = get_sparse_user_vector_for_tag(df_articles, "Computer Vision",
    corpus_vectorized, n_articles=3)
scores = user_vector.dot(corpus_vectorized.transpose())
scores_array = scores.toarray()[0]
show_best_results(df_articles, scores_array)

1 [score = 0.418693705486635]: Exploring Google Cloud Vision API and Feature Demonstration With Python
2 [score = 0.41540991349082357]: Converting a Picture into an Excel File
3 [score = 0.40255251172556206]: Label Classification of WCE Images With High Accuracy Using a Small Amount of Labels@ICCVW2019
4 [score = 0.30645121016529214]: Image Recognition APIs: Google, Amazon, IBM, Microsoft, and more
5 [score = 0.2653857134745245]: Intro to Segmentation
6 [score = 0.2637600262925986]: Essential OpenCV Functions to Get You Started into Computer Vision
7 [score = 0.24645612900435893]: Introduction to Artificial Intelligence
8 [score = 0.24074545572762504]: Google Objectron — A giant leap for the 3D object detection
9 [score = 0.23638084902772483]: Image Creation for Non-Artists (OpenCV Project Walkthrough)
10 [score = 0.22673661443732632]: How we built an easy-to-use image segmentation tool with transfer learning


In [13]:
# suppose the user has read three articles about Reinforcement Learning
user_vector, _ = get_sparse_user_vector_for_tag(df_articles, "Reinforcement Learning",
    corpus_vectorized, n_articles=3)
scores = user_vector.dot(corpus_vectorized.transpose())
scores_array = scores.toarray()[0]
show_best_results(df_articles, scores_array)

1 [score = 0.5819891692882346]: Mario’s Gym Routine
2 [score = 0.5629023964421458]: Beyond DQN/A3C: A Survey in Advanced Reinforcement Learning
3 [score = 0.4962385394805791]: A Brief History of the Beer Game
4 [score = 0.4215008988381338]: How do you get to Carnegie Hall?
5 [score = 0.4178575355580801]: How Entrepreneurs Can Thrive in a New Era of Uncertainty
6 [score = 0.4177584899340464]: Tools Of The Best: Software Development Edition
7 [score = 0.41171863437010386]: RL — Model-based Reinforcement Learning
8 [score = 0.4091304250429925]: Crash Course: Reinforcement Learning
9 [score = 0.4056764399626751]: Big Data’s Role in Creating Customer-Centric Business Intelligence
10 [score = 0.40349347709761546]: Creating a Tic-Tac-Toe game with a Q-learning AI which masters the game


It looks like mileage does vary! Computer vision and reinforcement learning worked better